# 10 · Feedback loop: fine-tune the segmenter on corrected cases

Week 9 of NeuroLens. The model in production (v1, notebook 07) is fine-tuned on BraTS train plus the cases a doctor
corrected. Here the corrections are **simulated**: the 40 validation cases where v1 did worst (`ml/data/feedback_sim.csv`),
with their expert BraTS masks standing in for a doctor's fix. Real corrections exported by the API use the same layout
(`--feedback-dir`).

1. Clone the repo and install the package (+ MONAI).
2. Run `python -m neurolens_ml.finetune_seg` with `ml/configs/seg_finetune.yaml`: start from v1, lr 5e-5, corrected
   cases repeated 10× per epoch. At each check both groups are measured; the kept epoch is the one that learned the
   corrections best while staying within 0.005 mean Dice of v1 on the other 148 val cases. If no epoch beats v1 on
   the corrections, v1 is kept. (The first run, fb1, used lr 2e-5, 5× and "must beat v1 on the other cases": v1 was kept.)
3. Show v1 vs v2 on the selection cases and on the corrected cases.

**Kaggle setup:** GPU T4 x2 (one is used), Internet on, inputs `dschettler8845/brats-2021-task1` and the output of the
`07_segmentation` notebook (it holds `segmenter_segresnet.pt`). Run with *Save Version → Save & Run All* (about 2 h),
then download everything from *Output*.

The test split is not opened here; old vs new on test is Day 3 (notebook 11).

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
CONFIG = "ml/configs/seg_finetune.yaml"
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data, checkpoint, --smoke)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# --no-deps: keep Kaggle's own torch and numpy; MONAI and nibabel need nothing else
missing = [p for p in ["neurolens_ml", "monai", "nibabel"] if importlib.util.find_spec(p) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if pkgs := [p for p in missing if p != "neurolens_ml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs], check=True)

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("repo commit:", commit, "| GPU:", gpu, "| CPUs:", os.cpu_count(), "| free disk in /tmp:", shutil.disk_usage("/tmp").free // 10**9, "GB")

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.finetune_seg", "--config", str(REPO_DIR / CONFIG), "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
# Stream the log into the notebook output, line by line
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "fine-tuning failed, see the log above"

In [ ]:
import pandas as pd
import yaml
from IPython.display import Image, display

name = yaml.safe_load((REPO_DIR / CONFIG).read_text())["name"]
metrics = json.loads((OUT_DIR / f"metrics_seg_{name}.json").read_text())
metrics["repo_commit"], metrics["gpu"] = commit, gpu  # which code and machine made this model
(OUT_DIR / f"metrics_seg_{name}.json").write_text(json.dumps(metrics, indent=2))
print("best epoch:", metrics["best_epoch"], "(0 = v1 kept) of", metrics["epochs_run"], "|", metrics["hours"], "h |", metrics["rule"])
table = pd.DataFrame({f"{v} {part}": metrics[v][part] for part in ("val_rest", "feedback") for v in ("v1", "v2")}).T
display(table)
display(Image(filename=str(OUT_DIR / f"curves_seg_{name}.png")))

In [ ]:
# Corrected cases: did the model learn the fixes? (They are training data now, so this shows learning, not generalization.)
fb = pd.read_csv(OUT_DIR / f"compare_feedback_{name}.csv")
for r in ("tc", "wt", "et"):
    fb[f"gain_{r}"] = fb[f"{r}_v2"] - fb[f"{r}_v1"]
fb[["id", "tc_v1", "tc_v2", "wt_v1", "wt_v2", "et_v1", "et_v2"]].round(3).sort_values("wt_v1").head(10)